# 🏀 NBA Wins Above Replacement (WAR)

## A Father-Son Data Science Project

This notebook calculates **Wins Above Replacement (WAR)** for NBA players using a technique called **Adjusted Plus-Minus regression**.

The big idea: instead of just looking at how well a team does when a player is on the court, we use math to *separate out* each player's individual contribution — controlling for who their teammates and opponents are.

By the end, we'll have a number for every player: "If this player were replaced by a replacement-level player (a fringe roster guy), how many wins would the team lose?"

---

**Sections:**
1. Setup & Data Collection
2. Parse Play-by-Play into Stints
3. Build the Stint Matrix
4. The Tiny Example (Teaching Moment)
5. Run the Ridge Regression
6. Convert to WAR
7. Validation & Visualization
8. Win Probability Leverage (Bonus)
9. Summary & Next Steps

## Install Dependencies

Run this cell first if you haven't installed the required libraries yet. It's safe to run even if they're already installed.

In [ ]:
!pip install nba_api pandas numpy scikit-learn matplotlib plotly tqdm scipy pyarrow

## Configuration

These are the main settings for the notebook. You can change them here without touching any other code.

- **SEASON**: Which NBA season to analyze
- **N_GAMES**: Set to a small number like `50` to do a quick test run. Use `None` for the full season (takes ~30 minutes)
- **REPLACEMENT_LEVEL**: How many points per 100 possessions below average a "replacement" player is. -2.0 is standard
- **CACHE_DIR**: Where to save downloaded data so we don't have to re-download it
- **FORCE_REFRESH**: Set to `True` to re-download data even if cache exists

In [ ]:
# === CONFIGURATION ===
SEASON = "2024-25"
N_GAMES = None           # Set to an integer to limit (e.g., 200 for testing)
REPLACEMENT_LEVEL = -2.0 # Per 100 possessions, relative to average
CACHE_DIR = "./nba_data_cache"
FORCE_REFRESH = False    # Set True to re-pull data even if cache exists

print(f"Season: {SEASON}")
print(f"N_GAMES: {N_GAMES if N_GAMES else 'Full season'}")
print(f"Replacement Level: {REPLACEMENT_LEVEL} pts/100 poss")
print(f"Cache directory: {CACHE_DIR}")
print(f"Force refresh: {FORCE_REFRESH}")

---
## Section 1: Setup & Data Collection

First, we import all the tools (libraries) we'll need. Think of imports like gathering your equipment before a game.

Then we download data from the NBA's official stats API. The NBA website (stats.nba.com) has an API — a way for programs to request data — and a library called `nba_api` makes it easy to use from Python.

We'll download:
1. A list of all games this season
2. Play-by-play data for each game (every single event: shots, fouls, substitutions)
3. Player info (so we can match player IDs to names)
4. Team standings (to validate our results later)

### 1a. Imports

Loading all our tools. `pandas` is for data tables, `numpy` is for math, `sklearn` is for machine learning, `scipy` handles sparse matrices (big grids where most values are zero), and `nba_api` talks to the NBA's servers.

In [ ]:
import os
import time
import math
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from scipy import sparse
from scipy.sparse import csr_matrix
from scipy.stats import norm
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.metrics import r2_score

from nba_api.stats.endpoints import (
    leaguegamefinder,
    playbyplayv2,
    gamerotation,
    commonallplayers,
    leaguestandings,
    boxscoretraditionalv2,
)
from nba_api.stats.static import teams as nba_teams_static

# Create cache directory if it doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)

print("All imports successful!")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

### 1b. Helper Functions for Robust API Calls

The NBA API sometimes fails — it might be busy, or our requests might come too fast. This helper function automatically retries up to 3 times if something goes wrong, waiting a bit longer each time (that's called "exponential backoff"). It's like knocking on a door: if no one answers, you wait a few seconds and try again.

In [ ]:
def api_call_with_retry(endpoint_class, max_retries=3, sleep_base=0.6, **kwargs):
    """Call an nba_api endpoint with automatic retry on failure."""
    for attempt in range(max_retries):
        try:
            time.sleep(sleep_base)  # Always sleep between calls to avoid rate limiting
            result = endpoint_class(**kwargs)
            return result
        except Exception as exc:
            wait_time = sleep_base * (2 ** attempt)  # Exponential backoff: 0.6, 1.2, 2.4 seconds
            if attempt < max_retries - 1:
                print(f"  API error (attempt {attempt+1}/{max_retries}): {exc}. Retrying in {wait_time:.1f}s...")
                time.sleep(wait_time)
            else:
                print(f"  API call failed after {max_retries} attempts: {exc}")
                return None
    return None

print("Helper function defined.")

### 1c. Pull Player Info

Player IDs in the NBA data are numbers like `2544` (that's LeBron James). We need a lookup table to convert IDs to names.

In [ ]:
%%time
players_cache_path = os.path.join(CACHE_DIR, "players.parquet")

if not FORCE_REFRESH and os.path.exists(players_cache_path):
    players_df = pd.read_parquet(players_cache_path)
    print(f"Loaded player info from cache: {len(players_df)} players")
else:
    print("Pulling player info from NBA API...")
    result = api_call_with_retry(
        commonallplayers.CommonAllPlayers,
        is_only_current_season=1,
        league_id="00",
        season=SEASON
    )
    if result is not None:
        players_df = result.get_data_frames()[0]
        players_df.to_parquet(players_cache_path, index=False)
        print(f"Pulled {len(players_df)} players")
    else:
        print("Could not pull player info, creating empty DataFrame")
        players_df = pd.DataFrame(columns=['PERSON_ID', 'DISPLAY_FIRST_LAST', 'TEAM_ABBREVIATION'])

# Build quick lookup dictionaries
player_id_to_name = dict(zip(players_df['PERSON_ID'].astype(int), players_df['DISPLAY_FIRST_LAST']))
player_id_to_team = dict(zip(players_df['PERSON_ID'].astype(int), players_df.get('TEAM_ABBREVIATION', players_df.get('TEAM_CITY', 'UNK'))))

print(f"Sample: {list(player_id_to_name.items())[:3]}")

### 1d. Pull Team Standings

We'll use the actual team win totals at the end to validate our model. If our WAR numbers are any good, teams with more total WAR should have more actual wins.

In [ ]:
%%time
standings_cache_path = os.path.join(CACHE_DIR, "standings.parquet")

if not FORCE_REFRESH and os.path.exists(standings_cache_path):
    standings_df = pd.read_parquet(standings_cache_path)
    print(f"Loaded standings from cache.")
else:
    print("Pulling team standings...")
    result = api_call_with_retry(
        leaguestandings.LeagueStandings,
        season=SEASON,
        season_type="Regular Season",
        league_id="00"
    )
    if result is not None:
        standings_df = result.get_data_frames()[0]
        standings_df.to_parquet(standings_cache_path, index=False)
    else:
        print("Could not pull standings.")
        standings_df = pd.DataFrame()

if len(standings_df) > 0:
    print(standings_df[['TeamName', 'TeamAbbreviation', 'WINS', 'LOSSES']].head(10).to_string(index=False))

### 1e. Get List of All Games

Before downloading play-by-play, we need to know all the game IDs for the season. `LeagueGameFinder` gives us a list of every game played.

In [ ]:
%%time
games_cache_path = os.path.join(CACHE_DIR, "games_list.parquet")

if not FORCE_REFRESH and os.path.exists(games_cache_path):
    games_df = pd.read_parquet(games_cache_path)
    print(f"Loaded game list from cache: {len(games_df)} game records")
else:
    print("Pulling game list from NBA API...")
    result = api_call_with_retry(
        leaguegamefinder.LeagueGameFinder,
        season_nullable=SEASON,
        league_id_nullable="00",
        season_type_nullable="Regular Season"
    )
    if result is not None:
        games_df = result.get_data_frames()[0]
        games_df.to_parquet(games_cache_path, index=False)
        print(f"Pulled {len(games_df)} game records")
    else:
        print("Could not pull game list.")
        games_df = pd.DataFrame()

# Each game appears twice (once per team), so deduplicate by GAME_ID
unique_game_ids = games_df['GAME_ID'].unique().tolist() if len(games_df) > 0 else []

# Optionally limit to N_GAMES for faster testing
if N_GAMES is not None:
    unique_game_ids = unique_game_ids[:N_GAMES]
    print(f"Limited to {N_GAMES} games for testing")

print(f"Total unique games to process: {len(unique_game_ids)}")
if len(unique_game_ids) > 0:
    print(f"Sample game IDs: {unique_game_ids[:5]}")

### 1f. Download Game Rotation Data

Instead of trying to figure out lineups from play-by-play events (which is tricky), we use the `gamerotation` endpoint. It directly tells us: "Player X was on the court from minute 3:45 to minute 8:20 in Q1." This is much more reliable.

This loop is the slowest part of the notebook. With the full season (~1,230 games) and a 0.6-second sleep, it takes about 12-15 minutes. With `N_GAMES = 200`, about 2-3 minutes.

In [ ]:
%%time
rotation_cache_path = os.path.join(CACHE_DIR, f"rotations_{SEASON.replace('-','_')}.parquet")

if not FORCE_REFRESH and os.path.exists(rotation_cache_path):
    all_rotations_df = pd.read_parquet(rotation_cache_path)
    print(f"Loaded rotation data from cache: {len(all_rotations_df)} records across {all_rotations_df['GAME_ID'].nunique()} games")
else:
    print(f"Downloading rotation data for {len(unique_game_ids)} games...")
    rotation_records = []
    failed_games = []

    for idx, game_id in enumerate(tqdm(unique_game_ids, desc="Downloading rotations")):
        result = api_call_with_retry(
            gamerotation.GameRotation,
            game_id=game_id
        )
        if result is None:
            failed_games.append(game_id)
            continue
        try:
            frames = result.get_data_frames()
            # frames[0] = away team rotations, frames[1] = home team rotations
            for team_idx, team_label in enumerate(['AWAY', 'HOME']):
                if team_idx < len(frames) and len(frames[team_idx]) > 0:
                    df = frames[team_idx].copy()
                    df['GAME_ID'] = game_id
                    df['TEAM_SIDE'] = team_label
                    rotation_records.append(df)
        except Exception as e:
            failed_games.append(game_id)
            print(f"  Error parsing game {game_id}: {e}")

    if rotation_records:
        all_rotations_df = pd.concat(rotation_records, ignore_index=True)
        all_rotations_df.to_parquet(rotation_cache_path, index=False)
        print(f"\nDownloaded rotations: {len(all_rotations_df)} records")
        if failed_games:
            print(f"Failed games ({len(failed_games)}): {failed_games[:10]}")
    else:
        print("No rotation data downloaded.")
        all_rotations_df = pd.DataFrame()

print(f"Rotation data columns: {list(all_rotations_df.columns) if len(all_rotations_df) > 0 else 'N/A'}")

### 1g. Download Play-by-Play Data

We also need the play-by-play to get the score at each moment in the game. This tells us how many points were scored during each stint.

In [ ]:
%%time
pbp_cache_path = os.path.join(CACHE_DIR, f"pbp_{SEASON.replace('-','_')}.parquet")

if not FORCE_REFRESH and os.path.exists(pbp_cache_path):
    all_pbp_df = pd.read_parquet(pbp_cache_path)
    print(f"Loaded PBP from cache: {len(all_pbp_df)} events across {all_pbp_df['GAME_ID'].nunique()} games")
else:
    print(f"Downloading play-by-play for {len(unique_game_ids)} games...")
    pbp_records = []
    failed_pbp = []

    for game_id in tqdm(unique_game_ids, desc="Downloading PBP"):
        result = api_call_with_retry(
            playbyplayv2.PlayByPlayV2,
            game_id=game_id
        )
        if result is None:
            failed_pbp.append(game_id)
            continue
        try:
            df = result.get_data_frames()[0]
            pbp_records.append(df)
        except Exception as e:
            failed_pbp.append(game_id)

    if pbp_records:
        all_pbp_df = pd.concat(pbp_records, ignore_index=True)
        all_pbp_df.to_parquet(pbp_cache_path, index=False)
        print(f"Downloaded PBP: {len(all_pbp_df)} events")
        if failed_pbp:
            print(f"Failed games: {len(failed_pbp)}")
    else:
        print("No PBP data downloaded.")
        all_pbp_df = pd.DataFrame()

print(f"PBP shape: {all_pbp_df.shape if len(all_pbp_df) > 0 else 'empty'}")

---
## Section 2: Parse Play-by-Play into Stints

This is the most complex data engineering step. A **stint** is a stretch of game time where the same 10 players are on the court — 5 per team, no changes.

Here's the plan:
1. Use the `gamerotation` data to know exactly when each player was on/off the court (it gives us start/end times in tenths of seconds)
2. Find every moment where a substitution happened (a player entered or left)
3. Each gap between substitutions is a stint
4. For each stint, look up the score at the start and end from play-by-play

The rotation data uses `IN_TIME_REAL` and `OUT_TIME_REAL` — these are measured in tenths of a second from the start of the game (so 2400 = 4 minutes into Q1, since a quarter is 720 seconds = 7200 tenths).

### 2a. Parse Score Timeline from Play-by-Play

First, let's extract a score timeline from play-by-play. For each scoring event, we'll record the game clock time (converted to seconds from tip-off) and the running score. This lets us look up "what was the score at time T?" for any stint.

In [ ]:
def pctimestring_to_seconds(pctimestring, period):
    """Convert NBA clock string like '5:30' in period 3 to total seconds from tip-off."""
    try:
        parts = str(pctimestring).strip().split(':')
        minutes = int(parts[0])
        seconds = float(parts[1])
        clock_seconds = minutes * 60 + seconds
        # Regular periods are 12 min each; OT periods are 5 min each
        if period <= 4:
            period_start = (period - 1) * 720
            return period_start + (720 - clock_seconds)
        else:
            ot_period = period - 4
            period_start = 4 * 720 + (ot_period - 1) * 300
            return period_start + (300 - clock_seconds)
    except:
        return None

def build_score_timeline(game_pbp):
    """
    Build a score timeline for a single game.
    Returns a list of (time_seconds, home_score, away_score) tuples, sorted by time.
    """
    timeline = [(0, 0, 0)]  # Start of game
    for _, row in game_pbp.iterrows():
        if pd.isna(row.get('SCORE')) or str(row.get('SCORE', '')).strip() == '':
            continue
        try:
            score_str = str(row['SCORE'])
            # Format is typically 'away - home' e.g. '45 - 52'
            parts = score_str.split(' - ')
            away_score = int(parts[0].strip())
            home_score = int(parts[1].strip())
            t = pctimestring_to_seconds(row.get('PCTIMESTRING', '12:00'), int(row.get('PERIOD', 1)))
            if t is not None:
                timeline.append((t, home_score, away_score))
        except:
            continue
    timeline.sort(key=lambda x: x[0])
    return timeline

def get_score_at_time(timeline, target_time):
    """Look up home_score, away_score at a given time using the timeline."""
    home, away = 0, 0
    for t, h, a in timeline:
        if t <= target_time:
            home, away = h, a
        else:
            break
    return home, away

print("Score timeline functions defined.")

### 2b. Parse Rotation Data into Stints

Now the main event. For each game:
1. Get all the "player IN" and "player OUT" times from rotation data
2. Find all the unique time breakpoints where the lineup changed
3. For each interval between breakpoints, figure out who was on the court
4. Record the score change during that stint

The `IN_TIME_REAL` and `OUT_TIME_REAL` columns in the rotation data are in tenths of a second. We divide by 10 to get seconds.

In [ ]:
%%time
stints_cache_path = os.path.join(CACHE_DIR, f"stints_{SEASON.replace('-','_')}.parquet")

if not FORCE_REFRESH and os.path.exists(stints_cache_path):
    stints_df = pd.read_parquet(stints_cache_path)
    print(f"Loaded stints from cache: {len(stints_df)} stints")
else:
    if len(all_rotations_df) == 0 or len(all_pbp_df) == 0:
        print("ERROR: Missing rotation or PBP data. Cannot build stints.")
        stints_df = pd.DataFrame()
    else:
        game_ids_with_both = set(all_rotations_df['GAME_ID'].unique()) & set(all_pbp_df['GAME_ID'].unique())
        print(f"Games with both rotation and PBP data: {len(game_ids_with_both)}")

        all_stints = []

        for game_id in tqdm(list(game_ids_with_both), desc="Parsing stints"):
            try:
                game_rot = all_rotations_df[all_rotations_df['GAME_ID'] == game_id].copy()
                game_pbp = all_pbp_df[all_pbp_df['GAME_ID'] == game_id].copy()

                # Build score timeline
                score_timeline = build_score_timeline(game_pbp)

                home_rot = game_rot[game_rot['TEAM_SIDE'] == 'HOME'].copy()
                away_rot = game_rot[game_rot['TEAM_SIDE'] == 'AWAY'].copy()

                if len(home_rot) == 0 or len(away_rot) == 0:
                    continue

                # Get home and away team IDs
                home_team_id = home_rot['TEAM_ID'].iloc[0] if 'TEAM_ID' in home_rot.columns else 0
                away_team_id = away_rot['TEAM_ID'].iloc[0] if 'TEAM_ID' in away_rot.columns else 0

                # Convert rotation times from tenths of seconds to seconds
                for df in [home_rot, away_rot]:
                    df['in_sec'] = df['IN_TIME_REAL'].astype(float) / 10.0
                    df['out_sec'] = df['OUT_TIME_REAL'].astype(float) / 10.0

                # All unique time breakpoints
                breakpoints = sorted(set(
                    home_rot['in_sec'].tolist() + home_rot['out_sec'].tolist() +
                    away_rot['in_sec'].tolist() + away_rot['out_sec'].tolist()
                ))

                for i in range(len(breakpoints) - 1):
                    t_start = breakpoints[i]
                    t_end = breakpoints[i + 1]
                    duration = t_end - t_start

                    if duration < 10:  # Skip stints shorter than 10 seconds
                        continue

                    # Midpoint to check who is on court
                    t_mid = (t_start + t_end) / 2

                    home_players = home_rot[
                        (home_rot['in_sec'] <= t_mid) & (home_rot['out_sec'] > t_mid)
                    ]['PERSON_ID'].tolist()

                    away_players = away_rot[
                        (away_rot['in_sec'] <= t_mid) & (away_rot['out_sec'] > t_mid)
                    ]['PERSON_ID'].tolist()

                    if len(home_players) != 5 or len(away_players) != 5:
                        continue  # Skip incomplete lineups

                    # Get scores
                    home_start, away_start = get_score_at_time(score_timeline, t_start)
                    home_end, away_end = get_score_at_time(score_timeline, t_end)

                    home_pts = home_end - home_start
                    away_pts = away_end - away_start

                    # Estimate possessions: duration / 14 (roughly 1 poss per 14 seconds)
                    possessions = max(duration / 14.0, 0.1)

                    point_diff = home_pts - away_pts
                    pts_per_100 = (point_diff / possessions) * 100

                    stint = {
                        'game_id': game_id,
                        'home_team_id': home_team_id,
                        'away_team_id': away_team_id,
                        't_start': t_start,
                        't_end': t_end,
                        'duration_seconds': duration,
                        'home_player_1': home_players[0],
                        'home_player_2': home_players[1],
                        'home_player_3': home_players[2],
                        'home_player_4': home_players[3],
                        'home_player_5': home_players[4],
                        'away_player_1': away_players[0],
                        'away_player_2': away_players[1],
                        'away_player_3': away_players[2],
                        'away_player_4': away_players[3],
                        'away_player_5': away_players[4],
                        'home_points': home_pts,
                        'away_points': away_pts,
                        'possessions': possessions,
                        'point_differential': point_diff,
                        'points_per_100_poss': pts_per_100,
                    }
                    all_stints.append(stint)

            except Exception as e:
                print(f"Error parsing game {game_id}: {e}")
                continue

        if all_stints:
            stints_df = pd.DataFrame(all_stints)
            # Filter out stints with unrealistic point totals (data errors)
            stints_df = stints_df[
                (stints_df['home_points'] >= 0) &
                (stints_df['away_points'] >= 0) &
                (stints_df['home_points'] <= 30) &
                (stints_df['away_points'] <= 30)
            ].reset_index(drop=True)
            stints_df.to_parquet(stints_cache_path, index=False)
            print(f"Parsed {len(stints_df)} valid stints")
        else:
            print("No stints parsed.")
            stints_df = pd.DataFrame()

print(f"\nStints DataFrame shape: {stints_df.shape}")
if len(stints_df) > 0:
    print(stints_df.head(3).to_string())

### 2c. Stint Validation & Statistics

Let's check our work. We'll print summary stats and look at the distribution of stint lengths. If something went badly wrong, this is where we'd catch it.

In [ ]:
if len(stints_df) > 0:
    print("=== Stint Summary Statistics ===")
    print(f"Total stints: {len(stints_df):,}")
    print(f"Games covered: {stints_df['game_id'].nunique():,}")
    print(f"Average stint duration: {stints_df['duration_seconds'].mean():.1f} seconds")
    print(f"Median stint duration: {stints_df['duration_seconds'].median():.1f} seconds")
    print(f"Total home points: {stints_df['home_points'].sum():,}")
    print(f"Total away points: {stints_df['away_points'].sum():,}")
    print(f"Avg pts/100 poss: {stints_df['points_per_100_poss'].mean():.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Stint length histogram
    axes[0].hist(stints_df['duration_seconds'].clip(upper=300), bins=50, color='royalblue', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Duration (seconds)')
    axes[0].set_ylabel('Number of Stints')
    axes[0].set_title('Distribution of Stint Lengths')
    axes[0].axvline(stints_df['duration_seconds'].median(), color='red', linestyle='--', label=f"Median: {stints_df['duration_seconds'].median():.0f}s")
    axes[0].legend()

    # Point differential histogram
    axes[1].hist(stints_df['points_per_100_poss'].clip(-200, 200), bins=60, color='darkorange', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Points per 100 Possessions (Home minus Away)')
    axes[1].set_ylabel('Number of Stints')
    axes[1].set_title('Distribution of Stint Outcomes')
    axes[1].axvline(0, color='black', linestyle='-', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(CACHE_DIR, 'stint_distributions.png'), dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("No stint data available to validate.")

---
## Section 3: Build the Stint Matrix

Here's the key idea: we're setting up a big system of equations. Each stint tells us: "When these 5 home players were on court against these 5 away players, the home team outscored the away team by X points per 100 possessions." The regression will solve for each player's individual contribution.

We build a matrix **X** where:
- Each **row** is one stint
- Each **column** is one player
- A home player gets a **+1** in their column
- An away player gets a **-1** in their column
- Everyone else is **0**

So if LeBron is a home player in stint #47, `X[47, lebron_column] = +1`.

The target vector **y** is the point differential per 100 possessions for each stint.

We use a **sparse matrix** because most entries are 0 — out of 500 players, only 10 are on court at once, so 98% of the matrix is zeros. Storing only the non-zero values saves a ton of memory.

In [ ]:
if len(stints_df) == 0:
    print("No stint data. Skipping matrix construction.")
else:
    # Collect all unique player IDs
    player_cols = ['home_player_1','home_player_2','home_player_3','home_player_4','home_player_5',
                   'away_player_1','away_player_2','away_player_3','away_player_4','away_player_5']

    all_player_ids = pd.unique(stints_df[player_cols].values.ravel())
    all_player_ids = sorted([p for p in all_player_ids if pd.notna(p) and p != 0])

    # Create player index: player_id -> column number
    player_index = {int(pid): col_idx for col_idx, pid in enumerate(all_player_ids)}
    n_players = len(all_player_ids)
    n_stints = len(stints_df)

    print(f"Number of unique players: {n_players}")
    print(f"Number of stints: {n_stints:,}")
    print(f"Matrix density: {(10 / n_players * 100):.2f}% non-zero per row (10 players per stint)")

    # Build sparse matrix using COO format first (row, col, value triples), then convert to CSR
    rows, cols, vals = [], [], []

    home_cols = ['home_player_1','home_player_2','home_player_3','home_player_4','home_player_5']
    away_cols = ['away_player_1','away_player_2','away_player_3','away_player_4','away_player_5']

    for stint_idx, row in stints_df.iterrows():
        stint_num = stint_idx  # row index in the matrix
        for col_name in home_cols:
            pid = int(row[col_name])
            if pid in player_index:
                rows.append(stint_num)
                cols.append(player_index[pid])
                vals.append(1.0)  # Home player = +1
        for col_name in away_cols:
            pid = int(row[col_name])
            if pid in player_index:
                rows.append(stint_num)
                cols.append(player_index[pid])
                vals.append(-1.0)  # Away player = -1

    # Build CSR sparse matrix (Compressed Sparse Row - efficient for row operations)
    X = csr_matrix((vals, (rows, cols)), shape=(n_stints, n_players))

    # Target vector and weights
    y = stints_df['points_per_100_poss'].values
    w = stints_df['possessions'].values  # Longer stints get more weight

    # Clip extreme outliers in y (can happen with data errors)
    y = np.clip(y, -300, 300)

    print(f"\nX shape: {X.shape}")
    print(f"X stored elements (non-zeros): {X.nnz:,}")
    print(f"y shape: {y.shape}, range: [{y.min():.1f}, {y.max():.1f}]")
    print(f"w shape: {w.shape}, range: [{w.min():.2f}, {w.max():.2f}]")

### 3b. Player Minutes Calculation

We also need to know how many minutes each player actually played — this is needed for the WAR calculation later. More minutes = more chances to help your team.

In [ ]:
if len(stints_df) > 0:
    # Sum up playing time for each player (in seconds, then convert to minutes)
    player_minutes = {}

    for pid in all_player_ids:
        pid_int = int(pid)
        home_mask = (
            (stints_df['home_player_1'] == pid_int) |
            (stints_df['home_player_2'] == pid_int) |
            (stints_df['home_player_3'] == pid_int) |
            (stints_df['home_player_4'] == pid_int) |
            (stints_df['home_player_5'] == pid_int)
        )
        away_mask = (
            (stints_df['away_player_1'] == pid_int) |
            (stints_df['away_player_2'] == pid_int) |
            (stints_df['away_player_3'] == pid_int) |
            (stints_df['away_player_4'] == pid_int) |
            (stints_df['away_player_5'] == pid_int)
        )
        total_seconds = stints_df[home_mask | away_mask]['duration_seconds'].sum()
        player_minutes[pid_int] = total_seconds / 60.0

    print(f"Calculated minutes for {len(player_minutes)} players")

    # Quick sanity check: top players by minutes
    top_by_min = sorted(player_minutes.items(), key=lambda x: x[1], reverse=True)[:10]
    print("\nTop 10 players by minutes played:")
    for pid, mins in top_by_min:
        name = player_id_to_name.get(pid, f"ID:{pid}")
        print(f"  {name}: {mins:.0f} min")

---
## Section 4: The Tiny Example (Teaching Moment)

Before we run the real model on thousands of stints and hundreds of players, let's build a tiny example by hand. This is the most important section for understanding *what we're actually doing*.

Imagine a 3-on-3 league with just 6 players: Alice, Bob, Carlos (Team A) and Diana, Eve, Frank (Team B).

Each row in our matrix says: "In this stint, these players faced each other, and the home team won by X points per 100 possessions."

In [ ]:
import numpy as np

# Players: Alice=0, Bob=1, Carlos=2, Diana=3, Eve=4, Frank=5
# Each row: [Alice, Bob, Carlos, Diana, Eve, Frank]
# +1 = home team, -1 = away team

print("=== Toy Example: 6 Players, 6 Stints ===")
print()
print("Players: Alice(0), Bob(1), Carlos(2) vs Diana(3), Eve(4), Frank(5)")
print()

# Stint matrix: each row is a stint
# Columns: Alice, Bob, Carlos, Diana, Eve, Frank
X_toy = np.array([
    [ 1,  1,  1, -1, -1, -1],  # ABC vs DEF, home wins by 8
    [ 1,  1,  1, -1, -1, -1],  # ABC vs DEF again, home wins by 6
    [ 1,  1, -1, -1,  1, -1],  # ABE vs DCF rearranged... let me fix:
    # Alice+Bob+Diana vs Carlos+Eve+Frank
    [ 1,  1, -1,  1, -1, -1],  # Alice,Bob HOME + Diana HOME vs Carlos,Eve,Frank AWAY
    [-1,  1,  1, -1,  1, -1],  # Bob,Carlos,Eve home vs Alice,Diana,Frank away
    [ 1, -1,  1, -1,  1, -1],  # Alice,Carlos,Eve home vs Bob,Diana,Frank away
], dtype=float)

# Point differentials per 100 possessions (home minus away)
y_toy = np.array([8.0, 6.0, 12.0, 4.0, -2.0, 10.0])

print("Stint matrix X (rows=stints, cols=players):")
print("        Alice  Bob  Carlos  Diana  Eve  Frank")
for i, row in enumerate(X_toy):
    print(f"Stint {i+1}:  {int(row[0]):+d}    {int(row[1]):+d}    {int(row[2]):+d}     {int(row[3]):+d}    {int(row[4]):+d}    {int(row[5]):+d}    → y = {y_toy[i]:+.1f} pts/100")

print()
print("Reading stint 1: When Alice(+1), Bob(+1), Carlos(+1) played at home")
print("  against Diana(-1), Eve(-1), Frank(-1), the home team won by 8 pts/100 poss.")

### 4a. Solving with Ordinary Least Squares

Ordinary Least Squares (OLS) finds the player ratings that best explain all the observed outcomes. It's asking: "What value should each player have so that adding up the home players' values minus the away players' values gets as close as possible to the actual point differential?"

This is exactly what `np.linalg.lstsq` does — it solves the system of equations `X @ ratings = y`.

In [ ]:
# Solve using OLS (ordinary least squares)
ratings_ols, residuals, rank, sv = np.linalg.lstsq(X_toy, y_toy, rcond=None)

player_names = ['Alice', 'Bob', 'Carlos', 'Diana', 'Eve', 'Frank']
print("=== OLS Player Ratings ===")
for name, rating in zip(player_names, ratings_ols):
    print(f"  {name}: {rating:+.2f} pts/100 poss")

print()
print("Let's verify: in Stint 1 (Alice+Bob+Carlos vs Diana+Eve+Frank):")
stint1_pred = X_toy[0] @ ratings_ols
print(f"  Predicted differential: {stint1_pred:+.2f} pts/100")
print(f"  Actual differential:    {y_toy[0]:+.2f} pts/100")
print()
print("The model isn't perfect — it finds the BEST fit across ALL stints simultaneously.")

### 4b. The Problem: Correlated Players

What happens if two players **always play together**? Imagine Alice and Bob are glued together — every time Alice plays, Bob plays, and vice versa.

Then the model can't tell: is it Alice who's good? Or Bob? Or is it their combination? The math becomes unstable — there are infinite solutions. This is called **multicollinearity**.

In [ ]:
# Example: Alice and Bob always play together (identical columns)
X_corr = np.array([
    [ 1,  1,  1, -1, -1, -1],  # Alice+Bob always together
    [ 1,  1, -1, -1,  1, -1],
    [ 1,  1,  1, -1, -1, -1],
    [ 1,  1, -1,  1, -1, -1],
    [ 1,  1,  1, -1, -1, -1],
    [-1, -1,  1,  1, -1, -1],
], dtype=float)

y_corr = np.array([8.0, 6.0, 7.0, 4.0, 9.0, -5.0])

# Note: columns 0 and 1 (Alice, Bob) are identical — always +1 or -1 together
ratings_corr, _, rank_corr, _ = np.linalg.lstsq(X_corr, y_corr, rcond=None)

print("=== Correlated Case: Alice and Bob always play together ===")
print(f"Matrix rank: {rank_corr} (should be 6 for full rank, less means correlated)")
print()
for name, rating in zip(player_names, ratings_corr):
    print(f"  {name}: {rating:+.2f} pts/100 poss")

print()
print("Notice: The model assigns ratings, but they're unstable.")
print("If Alice gets +10 and Bob gets -5, OR Alice gets -5 and Bob gets +10,")
print("the combined effect is the same. The model can't distinguish them.")

### 4c. The Solution: Ridge Regression

Ridge regression adds a **penalty** for large ratings. It's saying: "Find ratings that explain the data well, BUT don't let any player's rating get too extreme."

Mathematically, it minimizes: `||X @ ratings - y||² + alpha * ||ratings||²`

The first term says "fit the data well." The second term (controlled by `alpha`) says "keep ratings small." When players are correlated, Ridge splits the credit evenly between them — which is a reasonable thing to do!

The `alpha` parameter controls the balance. High alpha = more shrinkage = more conservative (ratings pulled toward zero). Low alpha = less shrinkage = closer to OLS.

In [ ]:
from sklearn.linear_model import Ridge

print("=== Ridge Regression on Correlated Case ===")
print()

for alpha in [0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_corr, y_corr)
    print(f"Alpha={alpha:5.1f}: Alice={ridge.coef_[0]:+.2f}, Bob={ridge.coef_[1]:+.2f}, "
          f"Carlos={ridge.coef_[2]:+.2f}, Diana={ridge.coef_[3]:+.2f}, "
          f"Eve={ridge.coef_[4]:+.2f}, Frank={ridge.coef_[5]:+.2f}")

print()
print("Key insight: As alpha increases, ratings shrink toward zero.")
print("Alice and Bob get split credit (they look equal since they always play together).")
print("This is much better than the unstable OLS result!")

# Show the effect of alpha on the original well-conditioned case
print()
print("=== Effect of Alpha on Original (non-correlated) Case ===")
alphas_test = [0.01, 1.0, 10.0, 100.0, 1000.0]
alice_ratings = []
for alpha in alphas_test:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_toy, y_toy)
    alice_ratings.append(ridge.coef_[0])
    print(f"Alpha={alpha:6.2f}: Alice={ridge.coef_[0]:+.3f}")

print()
print("Higher alpha shrinks Alice's rating toward 0 — more conservative estimate.")

---
## Section 5: Run the Ridge Regression

Now we run the real model on all the actual NBA data. We use **RidgeCV** which automatically tries different `alpha` values and picks the best one using cross-validation.

Cross-validation works like this: hold out 20% of the stints, fit the model on the other 80%, and see how well it predicts the held-out stints. Do this multiple times with different held-out sets, then pick the alpha that predicts best on average.

The output is one rating number per player, in units of **points per 100 possessions above average** (adjusted for teammates and opponents).

In [ ]:
%%time
if len(stints_df) == 0:
    print("No stint data available. Skipping regression.")
    player_ratings_df = pd.DataFrame()
else:
    print("Running RidgeCV on full stint matrix...")
    print(f"Matrix shape: {X.shape}")

    alphas = [1, 10, 50, 100, 500, 1000, 5000, 10000]
    model = RidgeCV(alphas=alphas, store_cv_results=True)
    model.fit(X, y, sample_weight=w)

    print(f"\nBest alpha: {model.alpha_}")
    print(f"Model intercept: {model.intercept_:.4f}")
    print(f"Number of player coefficients: {len(model.coef_)}")

    # Build results DataFrame
    player_ratings_df = pd.DataFrame({
        'player_id': [int(pid) for pid in all_player_ids],
        'rating': model.coef_
    })

    # Add player names and team
    player_ratings_df['player_name'] = player_ratings_df['player_id'].map(player_id_to_name).fillna('Unknown')
    player_ratings_df['team'] = player_ratings_df['player_id'].map(player_id_to_team).fillna('UNK')

    # Add minutes played
    player_ratings_df['minutes_played'] = player_ratings_df['player_id'].map(player_minutes).fillna(0)

    # Sort by rating
    player_ratings_df = player_ratings_df.sort_values('rating', ascending=False).reset_index(drop=True)

    # Filter to meaningful players (played at least 100 minutes)
    qualified_players = player_ratings_df[player_ratings_df['minutes_played'] >= 100].copy()

    print(f"\nTotal players with ratings: {len(player_ratings_df)}")
    print(f"Players with 100+ minutes: {len(qualified_players)}")

### 5b. Top and Bottom Players by Rating

Let's see who the model thinks are the best and worst players, measured in adjusted points per 100 possessions.

In [ ]:
if len(player_ratings_df) > 0:
    print("=== TOP 20 Players by Adjusted Rating ===")
    print(qualified_players.head(20)[['player_name', 'team', 'minutes_played', 'rating']].to_string(index=False))

    print()
    print("=== BOTTOM 20 Players by Adjusted Rating ===")
    print(qualified_players.tail(20)[['player_name', 'team', 'minutes_played', 'rating']].to_string(index=False))

### 5c. How Alpha Affects Player Ratings

Let's see how the top player's rating changes as we dial up or down the regularization. This builds intuition for what `alpha` is doing.

In [ ]:
if len(stints_df) > 0:
    alphas_experiment = [1, 10, 50, 100, 500, 1000, 5000, 10000]
    top_player_ratings = []
    second_player_ratings = []

    # Identify top 2 players from the main model
    if len(qualified_players) >= 2:
        top_pid = qualified_players.iloc[0]['player_id']
        second_pid = qualified_players.iloc[1]['player_id']
        top_col = player_index.get(int(top_pid), None)
        second_col = player_index.get(int(second_pid), None)
        top_name = qualified_players.iloc[0]['player_name']
        second_name = qualified_players.iloc[1]['player_name']

        for alpha in alphas_experiment:
            m = Ridge(alpha=alpha)
            m.fit(X, y, sample_weight=w)
            top_player_ratings.append(m.coef_[top_col] if top_col is not None else 0)
            second_player_ratings.append(m.coef_[second_col] if second_col is not None else 0)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.semilogx(alphas_experiment, top_player_ratings, 'o-', color='royalblue', label=top_name, linewidth=2)
        ax.semilogx(alphas_experiment, second_player_ratings, 's--', color='darkorange', label=second_name, linewidth=2)
        ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
        ax.set_xlabel('Alpha (regularization strength) — log scale')
        ax.set_ylabel('Adjusted Rating (pts/100 poss)')
        ax.set_title('Effect of Ridge Alpha on Player Ratings\n(Higher alpha = more shrinkage toward zero)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(CACHE_DIR, 'alpha_experiment.png'), dpi=100, bbox_inches='tight')
        plt.show()
        print(f"Best alpha from CV: {model.alpha_} (shown as vertical reference)")

---
## Section 6: Convert to WAR

A player's **rating** tells us how many points per 100 possessions they add above the **average** player. But WAR is measured against a **replacement-level** player — a fringe guy who could be called up from the G-League or signed off the street.

Why replacement level instead of average? Because average players are valuable and scarce. A team can always sign a replacement-level player. The question is: how much better than that minimum is this player?

The formula:
1. `marginal_value = rating - replacement_level` (how much better than replacement)
2. Scale by minutes (a player who plays 3,000 minutes helps more than one who plays 500)
3. Convert from "points per 100 possessions" to "wins" (about 30.89 point-margin-points = 1 win)

We set replacement level at **-2.0 pts/100** below average (configurable at the top of the notebook).

In [ ]:
if len(player_ratings_df) == 0:
    print("No player ratings available. Skipping WAR calculation.")
    war_df = pd.DataFrame()
else:
    war_df = qualified_players.copy()

    # Points per win (empirically about 30.89 for NBA, sometimes 30.5 is used)
    POINTS_PER_WIN = 30.89

    # Total player-minutes per team game: 5 players × 48 minutes = 240
    PLAYER_MINS_PER_GAME = 48 * 5

    # How many games in a full season
    GAMES_PER_SEASON = 82

    # Calculate marginal value above replacement
    war_df['marginal_value'] = war_df['rating'] - REPLACEMENT_LEVEL

    # WAR formula:
    # (marginal pts/100 poss) × (possessions played / 100) / (pts per win)
    # We approximate possessions played ≈ (minutes_played / 48) × (48*5/5) * (pace/10)
    # Simpler: war = marginal_value * (minutes_played / PLAYER_MINS_PER_GAME) * (GAMES_PER_SEASON / POINTS_PER_WIN)
    war_df['war'] = (
        war_df['marginal_value'] *
        (war_df['minutes_played'] / PLAYER_MINS_PER_GAME) *
        (GAMES_PER_SEASON / POINTS_PER_WIN)
    )

    # Sort by WAR
    war_df = war_df.sort_values('war', ascending=False).reset_index(drop=True)
    war_df['rank'] = war_df.index + 1

    print(f"WAR calculation complete for {len(war_df)} qualified players")
    print(f"\nReplacement level: {REPLACEMENT_LEVEL} pts/100 poss")
    print(f"Points per win: {POINTS_PER_WIN}")
    print(f"\n=== WAR Leaderboard (Top 30) ===")
    print(war_df.head(30)[['rank', 'player_name', 'team', 'minutes_played', 'rating', 'marginal_value', 'war']]
          .to_string(index=False, float_format=lambda x: f"{x:.2f}"))

### 6b. Save WAR Leaderboard

Let's save the results to a CSV so we can open it in Excel or share it.

In [ ]:
if len(war_df) > 0:
    output_path = "nba_war_2024_25.csv"
    war_df.to_csv(output_path, index=False)
    print(f"WAR leaderboard saved to {output_path}")

    print()
    print("=== 2024-25 NBA WAR Leaders ===")
    for _, row in war_df.head(15).iterrows():
        print(f"{int(row['rank']):2d}. {row['player_name']} ({row['team']}) — {row['war']:.1f} WAR")
else:
    print("No WAR data to save.")

---
## Section 7: Validation & Visualization

Numbers are only useful if we trust them. Let's check our work in several ways:
1. Do the top WAR players make intuitive sense (are they the players we'd expect)?
2. Do teams with more total WAR actually win more games?
3. How do our ratings compare to published metrics?

If the model is garbage, the team WAR vs. actual wins chart will just be a cloud of dots with no pattern. If it's good, we'll see a clear upward trend.

### 7a. Top 20 WAR Bar Chart

In [ ]:
if len(war_df) >= 5:
    top20 = war_df.head(20).copy()

    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(
        range(len(top20)),
        top20['war'],
        color=plt.cm.RdYlGn(np.linspace(0.4, 0.9, len(top20))),
        edgecolor='white',
        linewidth=0.5
    )

    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels([f"{row['player_name']} ({row['team']})" for _, row in top20.iterrows()], fontsize=10)
    ax.invert_yaxis()  # Best player at top

    ax.set_xlabel('Wins Above Replacement', fontsize=12)
    ax.set_title(f'Top 20 NBA Players by WAR — {SEASON}', fontsize=14, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.grid(axis='x', alpha=0.3)

    # Add value labels
    for i, (_, row) in enumerate(top20.iterrows()):
        ax.text(row['war'] + 0.1, i, f"{row['war']:.1f}", va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(CACHE_DIR, 'top20_war.png'), dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("Not enough data for WAR chart.")

### 7b. Team WAR vs. Actual Wins

The big validation test: if we sum up all the WAR for players on each team, it should correlate with the team's actual win total. This is the acid test for whether our model captures something real.

In [ ]:
if len(war_df) > 0 and len(standings_df) > 0:
    # Sum WAR by team
    team_war = war_df.groupby('team')['war'].sum().reset_index()
    team_war.columns = ['team_abbrev', 'total_war']

    # Get actual wins from standings
    # Try to find the right column name for team abbreviation
    abbrev_col = 'TeamAbbreviation' if 'TeamAbbreviation' in standings_df.columns else 'TEAM_ABBREVIATION'
    wins_col = 'WINS' if 'WINS' in standings_df.columns else 'W'

    if abbrev_col in standings_df.columns and wins_col in standings_df.columns:
        actual_wins = standings_df[[abbrev_col, wins_col]].copy()
        actual_wins.columns = ['team_abbrev', 'actual_wins']
        actual_wins['actual_wins'] = pd.to_numeric(actual_wins['actual_wins'], errors='coerce')

        # Merge
        team_validation = team_war.merge(actual_wins, on='team_abbrev', how='inner')

        if len(team_validation) >= 5:
            # Fit a line
            from numpy.polynomial import polynomial as P
            coeffs = np.polyfit(team_validation['total_war'], team_validation['actual_wins'], 1)
            line_x = np.linspace(team_validation['total_war'].min(), team_validation['total_war'].max(), 100)
            line_y = np.polyval(coeffs, line_x)

            r2 = r2_score(team_validation['actual_wins'], np.polyval(coeffs, team_validation['total_war']))

            fig, ax = plt.subplots(figsize=(10, 8))
            ax.scatter(team_validation['total_war'], team_validation['actual_wins'],
                      color='royalblue', s=100, zorder=5, alpha=0.8)
            ax.plot(line_x, line_y, 'r--', alpha=0.7, label=f'Trend line (R² = {r2:.2f})')

            for _, row in team_validation.iterrows():
                ax.annotate(row['team_abbrev'],
                           xy=(row['total_war'], row['actual_wins']),
                           xytext=(3, 3), textcoords='offset points', fontsize=8)

            ax.set_xlabel('Total Team WAR', fontsize=12)
            ax.set_ylabel('Actual Wins', fontsize=12)
            ax.set_title(f'Team WAR vs. Actual Wins — {SEASON}\nIf the model works, dots should follow the trend line', fontsize=13)
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(os.path.join(CACHE_DIR, 'team_war_vs_wins.png'), dpi=120, bbox_inches='tight')
            plt.show()

            print(f"R² = {r2:.3f} — {'Good correlation!' if r2 > 0.5 else 'Moderate correlation' if r2 > 0.3 else 'Weak correlation — model may need tuning'}")
            print(f"Teams in validation: {len(team_validation)}")
        else:
            print(f"Not enough teams to validate: {len(team_validation)}")
    else:
        print(f"Could not find expected columns in standings. Available: {list(standings_df.columns)}")
else:
    print("Missing war_df or standings_df for team validation.")

### 7c. Compare to Published Metrics (RAPTOR)

FiveThirtyEight published a metric called RAPTOR (Robust Algorithm using Player Tracking and On/Off Ratings). Let's try to pull their historical data and see if our ratings correlate with theirs. Note: FiveThirtyEight shut down in 2023, so this may only have data through the 2022-23 season.

In [ ]:
raptor_url = "https://raw.githubusercontent.com/fivethirtyeight/nba-player-advanced-metrics/master/nba-data-historical.csv"

try:
    raptor_df = pd.read_csv(raptor_url)
    print(f"Loaded RAPTOR data: {len(raptor_df)} rows")
    print(f"Columns: {list(raptor_df.columns)[:10]}")
    print(f"Seasons available: {sorted(raptor_df['season'].unique())[-5:] if 'season' in raptor_df.columns else 'unknown column'}")

    # Filter to most recent season available
    if 'season' in raptor_df.columns:
        latest_season = raptor_df['season'].max()
        raptor_latest = raptor_df[raptor_df['season'] == latest_season].copy()
        print(f"\nUsing RAPTOR season: {latest_season} ({len(raptor_latest)} players)")

        # Try to merge on player name
        if len(war_df) > 0:
            name_col = 'player_name' if 'player_name' in raptor_latest.columns else 'player'
            war_col = 'war_total' if 'war_total' in raptor_latest.columns else 'raptor_war'

            if name_col in raptor_latest.columns:
                merged = war_df.merge(
                    raptor_latest[[name_col, war_col]].dropna() if war_col in raptor_latest.columns else raptor_latest[[name_col]],
                    left_on='player_name', right_on=name_col, how='inner'
                )
                print(f"Matched {len(merged)} players with RAPTOR data")

                if len(merged) >= 10 and war_col in merged.columns:
                    fig, ax = plt.subplots(figsize=(8, 8))
                    ax.scatter(merged['war'], merged[war_col], alpha=0.6, color='royalblue', s=60)
                    r2_raptor = r2_score(merged[war_col], merged['war'])
                    ax.set_xlabel('Our WAR', fontsize=12)
                    ax.set_ylabel('RAPTOR WAR', fontsize=12)
                    ax.set_title(f'Our WAR vs. RAPTOR WAR (R² = {r2_raptor:.2f})', fontsize=13)
                    ax.grid(True, alpha=0.3)
                    plt.tight_layout()
                    plt.show()
                    print(f"Note: RAPTOR data is from season {latest_season}, not necessarily {SEASON}")
except Exception as e:
    print(f"Could not load RAPTOR data: {e}")
    print()
    print("That's okay! FiveThirtyEight stopped updating RAPTOR in 2023.")
    print("Instead, let's do a sanity check by looking at our top players:")
    if len(war_df) > 0:
        print()
        print("Do these names look right? Are these the players you'd expect to see at the top?")
        print(war_df.head(10)[['player_name', 'team', 'minutes_played', 'rating', 'war']].to_string(index=False))

### 7d. Distribution of Player Ratings

In a good model, most players should be clustered near zero (average), with a small number of elite players at the high end and a small number of bad players at the low end. This is called a normal distribution — like a bell curve.

In [ ]:
if len(qualified_players) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Ratings distribution
    axes[0].hist(qualified_players['rating'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(0, color='red', linestyle='--', label='League Average')
    axes[0].axvline(REPLACEMENT_LEVEL, color='orange', linestyle='--', label=f'Replacement Level ({REPLACEMENT_LEVEL})')
    axes[0].set_xlabel('Adjusted Rating (pts/100 poss)', fontsize=11)
    axes[0].set_ylabel('Number of Players', fontsize=11)
    axes[0].set_title('Distribution of Player Ratings', fontsize=13)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # WAR distribution
    axes[1].hist(war_df['war'], bins=40, color='darkorange', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='red', linestyle='--', label='0 WAR (replacement level)')
    axes[1].set_xlabel('WAR', fontsize=11)
    axes[1].set_ylabel('Number of Players', fontsize=11)
    axes[1].set_title('Distribution of WAR Values', fontsize=13)
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(CACHE_DIR, 'rating_distributions.png'), dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("Not enough data for distribution charts.")

### 7e. On/Off Comparison for Top Players

The **raw on/off differential** is simple: how does the point margin change when a player is on the court vs. off? The **adjusted rating** from our model controls for teammates and opponents.

The difference between these two numbers is the "teammate adjustment" — which is the whole point of building this model. A player on a great team might look good in raw on/off even if they're not contributing much individually.

In [ ]:
if len(stints_df) > 0 and len(war_df) >= 10:
    top10_pids = war_df.head(10)['player_id'].tolist()

    raw_on_off = {}
    for pid in top10_pids:
        pid_int = int(pid)
        # Stints where this player is on court (home or away)
        on_home = stints_df[(
            (stints_df['home_player_1'] == pid_int) |
            (stints_df['home_player_2'] == pid_int) |
            (stints_df['home_player_3'] == pid_int) |
            (stints_df['home_player_4'] == pid_int) |
            (stints_df['home_player_5'] == pid_int)
        )]
        on_away = stints_df[(
            (stints_df['away_player_1'] == pid_int) |
            (stints_df['away_player_2'] == pid_int) |
            (stints_df['away_player_3'] == pid_int) |
            (stints_df['away_player_4'] == pid_int) |
            (stints_df['away_player_5'] == pid_int)
        )]

        # When on home team, point_differential is from home's perspective = good for player
        # When on away team, negate (since differential is home minus away)
        on_pts = []
        on_poss = []
        if len(on_home) > 0:
            on_pts.extend(on_home['point_differential'].tolist())
            on_poss.extend(on_home['possessions'].tolist())
        if len(on_away) > 0:
            on_pts.extend((-on_away['point_differential']).tolist())
            on_poss.extend(on_away['possessions'].tolist())

        if sum(on_poss) > 0:
            raw_on_off[pid_int] = np.average(on_pts, weights=on_poss) / np.mean(on_poss) * 100
        else:
            raw_on_off[pid_int] = 0

    # Build comparison DataFrame
    compare_data = []
    for pid in top10_pids:
        pid_int = int(pid)
        row_data = war_df[war_df['player_id'] == pid_int].iloc[0]
        compare_data.append({
            'name': row_data['player_name'],
            'adjusted': row_data['rating'],
            'raw_on_off': raw_on_off.get(pid_int, 0)
        })

    compare_df = pd.DataFrame(compare_data)

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(compare_df))
    width = 0.35

    bars1 = ax.bar(x - width/2, compare_df['raw_on_off'], width, label='Raw On/Off', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, compare_df['adjusted'], width, label='Adjusted Rating (APM)', color='darkorange', alpha=0.8)

    ax.set_xlabel('')
    ax.set_ylabel('Points per 100 Possessions', fontsize=11)
    ax.set_title('Raw On/Off vs. Adjusted Rating for Top 10 WAR Players\n(Difference = teammate/opponent adjustment)', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(compare_df['name'], rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=10)
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(CACHE_DIR, 'on_off_comparison.png'), dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("Not enough data for on/off comparison.")

---
## Section 8: Win Probability Leverage (Bonus)

Here's a cool twist: not all points are equal. Scoring in a blowout means less than scoring in a close game. A player who excels in tight fourth-quarter situations is more valuable than one who pads stats in garbage time.

**Leverage** measures how much a stint changed the home team's probability of winning. If the game was already 30 points apart, the stint has low leverage. If it was tied with 2 minutes left, every possession is high leverage.

We'll re-run the regression with stints weighted by `possessions × leverage`, and see which players gain or lose value in the clutch-weighted model.

In [ ]:
def win_prob(margin, seconds_remaining, is_home):
    """
    Estimate probability that home team wins given:
    - margin: current score difference (home - away)
    - seconds_remaining: seconds left in the game
    - is_home: True if we're calculating from home team's perspective

    Formula based on a simple normal approximation:
    the final margin is roughly normal with mean = current_margin
    and std_dev proportional to sqrt(time_remaining).
    """
    home_advantage = 0.03 if is_home else -0.03
    if seconds_remaining <= 0:
        if margin > 0:
            return 1.0
        elif margin == 0:
            return 0.5
        else:
            return 0.0
    # 0.4 is an empirically tuned constant
    z = (margin / (math.sqrt(seconds_remaining) * 0.4)) + home_advantage
    return norm.cdf(z)

# Quick test
print("Win probability examples:")
print(f"  Tied with 5 min left (home): {win_prob(0, 300, True):.3f}")
print(f"  Up 10 with 2 min left (home): {win_prob(10, 120, True):.3f}")
print(f"  Down 5 with 1 min left (home): {win_prob(-5, 60, True):.3f}")
print(f"  Up 30 with 10 min left (home): {win_prob(30, 600, True):.3f}")
print(f"  Tied at tip-off (home): {win_prob(0, 2880, True):.3f}")

### 8b. Calculate Leverage for Each Stint

For each stint, we calculate:
1. Win probability at the **start** of the stint (given score and time at start)
2. Win probability at the **end** of the stint
3. Leverage = |win_prob_end - win_prob_start| (absolute change in win probability)

A stint in garbage time will barely move the win probability. A key possession in a tie game could swing it by 5-10%.

In [ ]:
%%time
leverage_cache_path = os.path.join(CACHE_DIR, f"stints_leverage_{SEASON.replace('-','_')}.parquet")

if not FORCE_REFRESH and os.path.exists(leverage_cache_path):
    stints_lev_df = pd.read_parquet(leverage_cache_path)
    print(f"Loaded leverage data from cache: {len(stints_lev_df)} stints")
elif len(stints_df) == 0 or len(all_pbp_df) == 0:
    print("Missing data for leverage calculation.")
    stints_lev_df = pd.DataFrame()
else:
    print("Calculating win probability leverage for each stint...")

    # Total game seconds (regular + OT)
    REGULATION_SECONDS = 4 * 720  # 2880 seconds

    stints_lev = stints_df.copy()
    leverages = []

    # Build score timelines for all games
    score_timelines = {}
    for game_id in tqdm(stints_lev['game_id'].unique(), desc="Building timelines"):
        game_pbp = all_pbp_df[all_pbp_df['GAME_ID'] == game_id]
        score_timelines[game_id] = build_score_timeline(game_pbp)

    print("Computing leverage per stint...")
    for _, row in tqdm(stints_lev.iterrows(), total=len(stints_lev), desc="Leverage"):
        game_id = row['game_id']
        t_start = row['t_start']
        t_end = row['t_end']

        timeline = score_timelines.get(game_id, [(0, 0, 0)])

        home_start, away_start = get_score_at_time(timeline, t_start)
        home_end, away_end = get_score_at_time(timeline, t_end)

        margin_start = home_start - away_start
        margin_end = home_end - away_end

        secs_remaining_start = max(REGULATION_SECONDS - t_start, 0)
        secs_remaining_end = max(REGULATION_SECONDS - t_end, 0)

        wp_start = win_prob(margin_start, secs_remaining_start, True)
        wp_end = win_prob(margin_end, secs_remaining_end, True)

        leverage = abs(wp_end - wp_start)
        leverages.append(leverage)

    stints_lev['leverage'] = leverages
    stints_lev['leverage_weight'] = stints_lev['possessions'] * (1 + stints_lev['leverage'])

    stints_lev.to_parquet(leverage_cache_path, index=False)
    stints_lev_df = stints_lev
    print(f"Leverage calculation complete. Mean leverage: {stints_lev_df['leverage'].mean():.4f}")

### 8c. Run Leverage-Weighted Regression

In [ ]:
%%time
if len(stints_lev_df) == 0 or len(player_ratings_df) == 0:
    print("Skipping leverage-weighted regression.")
    war_leverage_df = pd.DataFrame()
else:
    # Rebuild weight vector using leverage-weighted possessions
    w_leverage = stints_lev_df['leverage_weight'].values

    print("Running leverage-weighted RidgeCV...")
    model_lev = RidgeCV(alphas=alphas, store_cv_results=True)
    model_lev.fit(X, y, sample_weight=w_leverage)

    print(f"Best alpha (leverage model): {model_lev.alpha_}")

    # Build leverage WAR DataFrame
    war_leverage_df = war_df.copy()
    war_leverage_df['rating_leverage'] = [
        model_lev.coef_[player_index[int(pid)]] if int(pid) in player_index else 0
        for pid in war_leverage_df['player_id']
    ]
    war_leverage_df['marginal_leverage'] = war_leverage_df['rating_leverage'] - REPLACEMENT_LEVEL
    war_leverage_df['war_leverage'] = (
        war_leverage_df['marginal_leverage'] *
        (war_leverage_df['minutes_played'] / PLAYER_MINS_PER_GAME) *
        (GAMES_PER_SEASON / POINTS_PER_WIN)
    )
    war_leverage_df['war_diff'] = war_leverage_df['war_leverage'] - war_leverage_df['war']
    war_leverage_df = war_leverage_df.sort_values('war_diff', ascending=False)

    print("\n=== Clutch Performers (biggest WAR gain with leverage weighting) ===")
    print(war_leverage_df.head(10)[['player_name', 'team', 'war', 'war_leverage', 'war_diff']].to_string(index=False, float_format=lambda x: f"{x:.2f}"))

    print("\n=== Garbage Time Heroes (biggest WAR loss with leverage weighting) ===")
    print(war_leverage_df.tail(10)[['player_name', 'team', 'war', 'war_leverage', 'war_diff']].to_string(index=False, float_format=lambda x: f"{x:.2f}"))

### 8d. Scatter Plot: Standard WAR vs. Leverage-Weighted WAR

In [ ]:
if len(war_leverage_df) >= 20:
    fig = px.scatter(
        war_leverage_df,
        x='war',
        y='war_leverage',
        text='player_name',
        color='war_diff',
        color_continuous_scale='RdYlGn',
        title=f'Standard WAR vs. Leverage-Weighted WAR — {SEASON}',
        labels={'war': 'Standard WAR', 'war_leverage': 'Leverage-Weighted WAR', 'war_diff': 'Difference'},
        hover_data=['team', 'minutes_played'],
        width=850,
        height=700
    )
    # Add diagonal reference line (y=x means no change)
    min_val = min(war_leverage_df['war'].min(), war_leverage_df['war_leverage'].min())
    max_val = max(war_leverage_df['war'].max(), war_leverage_df['war_leverage'].max())
    fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
                  line=dict(color='gray', width=1, dash='dash'))
    fig.update_traces(textposition='top center', textfont_size=8)
    fig.update_layout(coloraxis_colorbar=dict(title='WAR Diff'))
    fig.show()

    print("Points ABOVE the diagonal: players who perform better in high-leverage situations (clutch)")
    print("Points BELOW the diagonal: players who perform better in low-leverage situations (garbage time)")
else:
    print("Not enough data for leverage scatter plot.")

---
## Section 9: Summary & Next Steps

### What We Built

We built a complete **Wins Above Replacement** system for the NBA from scratch using Python. Here's what we did:

1. **Downloaded real NBA data** from the NBA's official stats API — rotation data and play-by-play for every game
2. **Parsed the data into stints** — stretches where the same 10 players were on court
3. **Built a giant sparse matrix** where each row is a stint and each column is a player (+1 for home, -1 for away)
4. **Used Ridge regression** to find each player's individual contribution, controlling for teammates and opponents
5. **Converted ratings to WAR** by scaling by minutes played and the "points per win" conversion factor
6. **Validated the model** by checking if team WAR predicts team wins
7. **Added leverage weighting** to measure clutch performance

### What a 13-Year-Old Should Focus On

The conceptual core is in **Sections 3-6**. The key insight is:

- We can write a player's value as an **unknown variable**
- Every stint gives us an **equation** relating multiple players' values to an observed outcome
- With thousands of stints, we have way more equations than unknowns — so we solve for the best-fit values
- Ridge regression keeps the answers **stable** even when players' data is correlated

This is the same math used in Google's original PageRank algorithm, financial risk models, and all kinds of machine learning — just applied to basketball!

### Limitations

- **Sample size**: Some players have very little data (few minutes played), so their ratings are noisy
- **No defensive breakdown**: Our single rating mixes offense and defense
- **Context**: The model doesn't know about injuries, back-to-backs, altitude, or team strategy
- **Possessions estimate**: We approximate possessions as `duration / 14` instead of counting actual possessions

### Ideas for Extensions

1. **Offensive vs. Defensive WAR**: Use points scored (not point differential) for O-WAR, and points allowed for D-WAR
2. **Multi-year trends**: Pull 5 years of data and see how players' WAR changes over their career
3. **Playoff WAR**: Re-run the model on playoff data only — does playoff performance predict championship?
4. **Position breakdown**: Compare WAR distributions by position — are centers undervalued?
5. **Contract value**: Find the best WAR-per-dollar players (most underpaid)
6. **Predictive model**: Does this year's WAR predict next year's wins?

The NBA's own internal analytics teams use versions of this methodology — you've built something real!

In [ ]:
# Final summary output
print("=" * 50)
print(f"  2024-25 NBA WAR Analysis — Final Summary")
print("=" * 50)

if len(stints_df) > 0:
    print(f"\n  Data collected:")
    print(f"    Games processed: {stints_df['game_id'].nunique():,}")
    print(f"    Total stints: {len(stints_df):,}")
    print(f"    Players analyzed: {len(player_ratings_df):,}")

if len(war_df) > 0:
    print(f"\n  WAR Leaders:")
    for _, row in war_df.head(10).iterrows():
        print(f"    {int(row['rank']):2d}. {row['player_name']:<25s} ({row['team']}) — {row['war']:.1f} WAR")

print(f"\n  Configuration used:")
print(f"    Season: {SEASON}")
print(f"    Replacement level: {REPLACEMENT_LEVEL} pts/100 poss")
print(f"    N_GAMES: {N_GAMES if N_GAMES else 'Full season'}")

print("\n  Output files saved:")
print("    nba_war_2024_25.csv — Full WAR leaderboard")
print(f"    {CACHE_DIR}/ — Cached data (rotations, PBP, stints)")
print()
print("  Great work! Go explore the data and see if the numbers match your basketball intuition.")
print("=" * 50)